In [9]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from urllib.parse import urlparse, urlunparse
from rapidfuzz import fuzz
import re

CapIQ_data=pd.read_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\CapIQ_Web_Scraping_Output_87922-89129_Companies.xlsx")
CapIQ_data=CapIQ_data[(CapIQ_data['Website URL'].notna()) & (CapIQ_data['LinkedIN Profile'].isna())]
CapIQ_data.shape

(2, 16)

In [10]:
CapIQ_data

,Employer Name,City,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,LinkedIN URL,Executive Name,Executive Title,LinkedIN Profile
37,Hahn Systems,Indianapolis,423840 - Industrial Supplies Merchant Wholesalers,NaN,NaN,http://www.hahnsystems.com/,"8416 Zionsille Road, Indianapolis, IN 46268, USA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,Hallmark Homes Inc,Anderson,236115 - New Single-Family Housing Constructio...,NaN,Hallmark Homes is a prominent custom home buil...,http://www.hallmarkhomes.com/,"433 East 53rd Street, Anderson, IN 46013, USA",20.0,5.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import re
import pandas as pd
import time

# Setup Chrome options
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
service = Service(r"C:\Users\IlaBarshilia\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe")  # Replace with your actual path

# Initialize driver
driver = webdriver.Chrome(service=service, options=options)

def extract_contact_info_from_keywords(url):
    try:
        driver.get(url)
        time.sleep(3)

        keywords = ["Contact", "Contact Us", "Phone", "Tel", "Email"]
        contact_info = {"emails": [], "phones": [], "text_snippets": []}

        # Search for elements containing any of the keywords
        for keyword in keywords:
            elements = driver.find_elements(By.XPATH, f"//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{keyword.lower()}')]")
            for element in elements:
                try:
                    driver.execute_script("arguments[0].scrollIntoView(true);", element)
                    time.sleep(1)

                    # Get nearby text (parent or sibling)
                    parent = element.find_element(By.XPATH, "..")
                    snippet = parent.text

                    # Extract emails and phone numbers
                    emails = re.findall(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", snippet)
                    phones = re.findall(r"\+?\d[\d\s\-()]{7,}", snippet)

                    contact_info["emails"].extend(emails)
                    contact_info["phones"].extend(phones)
                    contact_info["text_snippets"].append(snippet)

                except Exception:
                    continue

        # Deduplicate results
        contact_info["emails"] = list(set(contact_info["emails"]))
        contact_info["phones"] = list(set(contact_info["phones"]))

        return contact_info

    except Exception as e:
        return {"error": str(e)}

# Apply function
CapIQ_data["Contact_Email"] = CapIQ_data["Website URL"].apply(lambda url: extract_contact_info_from_keywords(url).get("emails", []))
CapIQ_data["Contact_Phone"] = CapIQ_data["Website URL"].apply(lambda url: extract_contact_info_from_keywords(url).get("phones", []))

# Close driver
driver.quit()

CapIQ_data

,Employer Name,City,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,LinkedIN URL,Executive Name,Executive Title,LinkedIN Profile,contact_info,Contact_Email,Contact_Phone
37,Hahn Systems,Indianapolis,423840 - Industrial Supplies Merchant Wholesalers,NaN,NaN,http://www.hahnsystems.com/,"8416 Zionsille Road, Indianapolis, IN 46268, USA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'emails': ['sales@hahnsystems.com'], 'phones'...",[sales@hahnsystems.com],"[+1 800-201-4246\n, +1 800-201-4246]"
121,Hallmark Homes Inc,Anderson,236115 - New Single-Family Housing Constructio...,NaN,Hallmark Homes is a prominent custom home buil...,http://www.hallmarkhomes.com/,"433 East 53rd Street, Anderson, IN 46013, USA",20.0,5.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,"{'emails': [], 'phones': ['4 3', '4 ...",[],"[4 3, 4 2 2, 5 2, 4 2]"


In [24]:
CapIQ_data["contact_info"]

CapIQ_exploded = CapIQ_data.explode('contact_info')
CapIQ_exploded


,Employer Name,City,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,LinkedIN URL,Executive Name,Executive Title,LinkedIN Profile,contact_info
37,Hahn Systems,Indianapolis,423840 - Industrial Supplies Merchant Wholesalers,NaN,NaN,http://www.hahnsystems.com/,"8416 Zionsille Road, Indianapolis, IN 46268, USA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,emails
37,Hahn Systems,Indianapolis,423840 - Industrial Supplies Merchant Wholesalers,NaN,NaN,http://www.hahnsystems.com/,"8416 Zionsille Road, Indianapolis, IN 46268, USA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,phones
37,Hahn Systems,Indianapolis,423840 - Industrial Supplies Merchant Wholesalers,NaN,NaN,http://www.hahnsystems.com/,"8416 Zionsille Road, Indianapolis, IN 46268, USA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,text_snippets
121,Hallmark Homes Inc,Anderson,236115 - New Single-Family Housing Constructio...,NaN,Hallmark Homes is a prominent custom home buil...,http://www.hallmarkhomes.com/,"433 East 53rd Street, Anderson, IN 46013, USA",20.0,5.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,emails
121,Hallmark Homes Inc,Anderson,236115 - New Single-Family Housing Constructio...,NaN,Hallmark Homes is a prominent custom home buil...,http://www.hallmarkhomes.com/,"433 East 53rd Street, Anderson, IN 46013, USA",20.0,5.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,phones
121,Hallmark Homes Inc,Anderson,236115 - New Single-Family Housing Constructio...,NaN,Hallmark Homes is a prominent custom home buil...,http://www.hallmarkhomes.com/,"433 East 53rd Street, Anderson, IN 46013, USA",20.0,5.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,text_snippets


In [27]:
import pandas as pd
df=pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - Source Data\Acme DOT Jobs (from Sage 100).xlsx")
df.columns=df.iloc[0]
df=df[1:]
df.head()

,CustomerName,CustomerNo,DateCreated,JobDesc,JobNo,JobStatus,JobType,SalespersonNo,SortField
1,Ajax Paving Industries of Florida,0000049,2025-02-17 00:00:00,E1W36 Perm Signs - SR 66 From,E1W36PS,O,NDT,NaN,NaN
2,Ajax Paving Industries of Florida,0000049,2025-02-17 00:00:00,E1W77 (Non Contract) - SR 70 &,E1W77N,O,NDT,NaN,NaN
3,Hinson Electric,0000847,2025-03-12 00:00:00,E21T9 - Fast Response,E21T9,O,NDT,NaN,NaN
4,C W Roberts,0000249,2025-02-20 00:00:00,E3X17 - SR 297,E3X17,O,NDT,NaN,NaN
5,"RAM Construction Svcs of Michigan, Inc",0001321,2025-05-01 00:00:00,FL DOT Call 002 - Bridge Repa,E3X72,O,NDT,NaN,11488


In [30]:
df2 =pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - Source Data\FDOT Master Pay Items.xlsx")
df2.columns=df2.iloc[1]
df2=df2[2:]
df2.head()

1,Item Number & Item Desc. Combo,Acme Vlookup,Acme LOB,Item Number,Item Description,Unit Measure,Item Class,Spec Tech,Combine Flag,Is Obsolete,Valid Date,Obsolete Date,Reference Price
2,0 30 1-DESIGN COSTS FOR DESIGN BUILD PROJECT:...,NaN,NaN,0 30 1,DESIGN COSTS FOR DESIGN BUILD PROJECT: ESTIMAT...,LS/LS,9999,E,N,NaN,2015-08-24 00:00:00,NaN,NaN
3,"0 50 1-DESIGN - BUILD, RESURFACING",NaN,NaN,0 50 1,"DESIGN - BUILD, RESURFACING",LS/LS,07,T,N,NaN,2013-01-01 00:00:00,NaN,NaN
4,"0 50 2-DESIGN / BUILD, ROADWAY",NaN,NaN,0 50 2,"DESIGN / BUILD, ROADWAY",LS/LS,07,T,N,NaN,2013-01-01 00:00:00,NaN,NaN
5,"0 50 4-DESIGN / BUILD, BRIDGE CONSTRUCTION",NaN,NaN,0 50 4,"DESIGN / BUILD, BRIDGE CONSTRUCTION",LS/LS,10,T,N,NaN,2013-01-01 00:00:00,NaN,NaN
6,"0 50 5-DESIGN / BUILD, BUILDING / TOLL FACILI...",NaN,NaN,0 50 5,"DESIGN / BUILD, BUILDING / TOLL FACILITY/ REST...",LS/LS,10,T,N,NaN,2013-01-01 00:00:00,NaN,NaN
